# Python 8: concurrency

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-08-concurrency.ipynb)

A pipeline spends its time in two kinds of work. It **waits** for the network, the disk or a database, or it **computes**, by parsing, hashing or aggregating. Python handles the two differently. This chapter explains the Global Interpreter Lock, when threads help, when processes help, and how `asyncio` handles thousands of waits in one thread.

Every rule is stated before the code that shows it. The timing checks are deliberately loose, so they hold on a slow machine. If an `assert` fails, your model of that rule is wrong.
**Runtime:** Python 3.12 or later.

## The GIL: one thread runs Python at a time

CPython has a **Global Interpreter Lock**, the GIL. Only one thread executes Python bytecode at any moment. A thread that **waits** for I/O releases the GIL, so another thread can run while it waits. Threads therefore speed up waiting work. They do **not** speed up pure Python computation, because the threads take turns on the one lock.

For computation, use **processes**. Each process has its own interpreter and its own GIL, so the work runs on several cores. A library written in C that releases the GIL, such as NumPy or Polars, also computes in parallel inside one process.

Rule of thumb: **threads for waiting, processes for computing, asyncio for very many waits.**

### Where you'll use it

- **Choosing the tool:** a pipeline that calls ten APIs needs threads or asyncio. One that compresses or hashes large files needs processes or a native library.
- **Explaining slow jobs:** a Python loop over a million rows is slow partly because of the GIL. That is why engines such as Spark and Polars move the heavy work out of Python.

## Vocabulary: process, thread and coroutine

- A **process** is a running program with its own memory. Two processes do not share variables.
- A **thread** is one line of execution inside a process. Threads in one process share its memory.
- A **coroutine** is a function defined with `async def`. It can pause at `await` and let another coroutine run. All coroutines in one event loop share one thread.

Every Python program starts with one process and one main thread. The cell below checks that.

In [ ]:
import os
import threading

assert os.getpid() > 0                                  # the id of this process
assert threading.current_thread().name == "MainThread"
assert threading.active_count() >= 1

## Threads for I/O: `ThreadPoolExecutor`

`ThreadPoolExecutor` runs a function over many inputs with a pool of threads. It suits fetching many URLs, calling an API for each record, or reading many files.

Two ways to collect results differ in **order**:

- `pool.map(fn, items)` returns results in the **input** order, even when the calls finish out of order.
- `pool.submit(fn, item)` returns a future for each call. `as_completed(futures)` yields the futures in **completion** order, so the fastest result comes first.

The example below simulates a network call that waits 0.2 seconds. Ten calls one after another take about two seconds. With ten threads they take about 0.2 seconds.

### Where you'll use it

- **Calling many REST endpoints:** fetching one page per customer from an API.
- **Downloading files:** pulling hundreds of objects from S3 or GCS in parallel.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def fetch(page):
    time.sleep(0.2)                       # stands in for a network round trip
    return page * 10

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=10) as pool:
    results = list(pool.map(fetch, range(10)))
elapsed = time.perf_counter() - start

assert results == [n * 10 for n in range(10)]     # input order is preserved
assert elapsed < 1.0                              # far below the two seconds one-by-one would take

def fetch_after(delay, name):
    time.sleep(delay)
    return name

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(fetch_after, 0.3, "slow"), pool.submit(fetch_after, 0.05, "fast")]
    completion_order = [future.result() for future in as_completed(futures)]

assert completion_order == ["fast", "slow"]       # the quick call finished first

## Shared state needs a lock

Threads share memory. `counter += 1` is not one step. It reads the value, adds one and writes it back, and another thread can run between those steps and overwrite the result. The outcome then depends on timing, which is the worst kind of bug to find.

A `threading.Lock` makes a block of code run by one thread at a time. Put the whole read-modify-write inside `with lock:`. Even better, avoid sharing mutable state: return results from the workers, as `pool.map` does, and combine them in one place.

### Where you'll use it

- **Metrics and counters:** several workers adding to one row count must not lose updates.
- **Caches:** a shared token cache is refreshed by one thread while others read it.

In [ ]:
import threading

counter = {"value": 0}
lock = threading.Lock()

def bump(times):
    for _ in range(times):
        with lock:                         # one thread at a time through this block
            counter["value"] += 1

threads = [threading.Thread(target=bump, args=(10_000,)) for _ in range(4)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()                          # wait until the thread has finished

assert counter["value"] == 40_000

## Producer and consumer: `queue.Queue`

A `queue.Queue` passes items between threads safely. It handles the locking itself. A common design has one thread put work on the queue and several worker threads take it off. To stop the workers, the producer puts one **sentinel** value per worker. The sentinel `None` here means "no more work".

### Where you'll use it

- **Read, transform, write stages:** a reader thread feeds a parser pool, which feeds a single writer thread.
- **Rate control:** a bounded `Queue(maxsize=...)` makes the producer wait when the consumers fall behind.

In [ ]:
import queue

tasks = queue.Queue()
squares = []
squares_lock = threading.Lock()

def worker():
    while True:
        item = tasks.get()
        try:
            if item is None:               # sentinel: no more work for this thread
                return
            with squares_lock:
                squares.append(item * item)
        finally:
            tasks.task_done()

workers = [threading.Thread(target=worker) for _ in range(3)]
for thread in workers:
    thread.start()

for number in range(1, 6):
    tasks.put(number)
for _ in workers:
    tasks.put(None)                        # one sentinel for each worker
for thread in workers:
    thread.join()

assert sorted(squares) == [1, 4, 9, 16, 25]    # the order of completion is not fixed

## Processes for computation

`ProcessPoolExecutor` runs work in separate interpreters, so CPU-heavy code uses several cores. Each process has its own memory, so arguments and results are **copied** between processes. Send small inputs, such as file paths, and return small summaries.

The worker function must be importable by name, and the start-up code must sit behind an `if __name__ == "__main__":` guard. Notebook platforms do not always support that the same way on every system, so the example below is shown as a reference rather than run here.

```python
from concurrent.futures import ProcessPoolExecutor

def checksum(chunk):
    return sum(x * x for x in chunk)

if __name__ == "__main__":
    chunks = [range(i, i + 1_000_000) for i in range(0, 4_000_000, 1_000_000)]
    with ProcessPoolExecutor() as pool:
        partials = list(pool.map(checksum, chunks))
    total = sum(partials)
```

### Where you'll use it

- **CPU-heavy transforms:** hashing, compressing or parsing large files across all cores.
- **Isolation:** a worker process that crashes does not take down the parent job, which matters when you parse untrusted files.

## `asyncio` for many waits

`async def` defines a **coroutine function**. Calling it returns a **coroutine object** and runs nothing yet. `await` runs the coroutine and pauses the current one until it finishes. An **event loop** runs many coroutines that mostly wait, in one thread, with no locks.

Notebooks already run an event loop, so you can use `await` at the top level of a cell. In a script, the entry point is `asyncio.run(main())`.

### Where you'll use it

- **High fan-out I/O:** calling a service thousands of times, where each call mostly waits.
- **Async clients:** modern HTTP and database clients, such as `httpx` and `asyncpg`, are built on asyncio.

In [ ]:
import asyncio

async def call_api(endpoint):
    await asyncio.sleep(0.1)               # stands in for an HTTP request
    return f"{endpoint}: 200"

coroutine = call_api("/items/0")
assert asyncio.iscoroutine(coroutine)      # calling it ran none of the body
assert await coroutine == "/items/0: 200"  # await runs it to completion

async def fetch_all(endpoints):
    return await asyncio.gather(*(call_api(e) for e in endpoints))

endpoints = [f"/items/{n}" for n in range(50)]
start = time.perf_counter()
responses = await fetch_all(endpoints)
elapsed = time.perf_counter() - start

assert responses[0] == "/items/0: 200" and len(responses) == 50
assert elapsed < 1.0                       # 50 waits of 0.1 s overlap; they do not add up to 5 s

### Errors and timeouts in async code

`asyncio.gather` raises the first error it meets. Pass `return_exceptions=True` to get every result, and check each one for an exception instance. `asyncio.wait_for` cancels a call that takes longer than its timeout, and raises `asyncio.TimeoutError`.

In [ ]:
async def maybe_fail(n):
    await asyncio.sleep(0.01)
    if n == 2:
        raise ValueError("bad endpoint")
    return n

results = await asyncio.gather(*(maybe_fail(n) for n in range(4)), return_exceptions=True)
assert results[0] == 0 and results[3] == 3
assert isinstance(results[2], ValueError)  # the failure is a value, and the others still finished

async def slow_call():
    await asyncio.sleep(1.0)
    return "late"

try:
    await asyncio.wait_for(slow_call(), timeout=0.05)
except asyncio.TimeoutError:
    timed_out = True
else:
    timed_out = False
assert timed_out

## Limiting concurrency

Unlimited concurrency can overload the system you call, such as a database with a fixed number of connections or an API with a rate limit. An `asyncio.Semaphore` caps how many coroutines run at once. Each coroutine takes a slot with `async with` and gives it back when it leaves the block.

### Where you'll use it

- **Respecting rate limits:** an API that allows 10 requests in flight. A semaphore keeps the pipeline inside that limit.
- **Protecting databases:** a bounded pool stops a backfill from exhausting the database's connections.

In [ ]:
async def bounded_fetch(endpoints, limit):
    gate = asyncio.Semaphore(limit)
    in_flight = {"now": 0, "peak": 0}

    async def one(endpoint):
        async with gate:
            in_flight["now"] += 1
            in_flight["peak"] = max(in_flight["peak"], in_flight["now"])
            await asyncio.sleep(0.01)
            in_flight["now"] -= 1
            return endpoint

    results = await asyncio.gather(*(one(e) for e in endpoints))
    return results, in_flight["peak"]

results, peak = await bounded_fetch(list(range(20)), limit=4)
assert len(results) == 20
assert peak == 4                           # never more than four in flight at once

## Practice questions

1. Explain why `ThreadPoolExecutor` speeds up a loop of HTTP calls but does not speed up a loop that parses numbers.
2. Write `async def first_ok(endpoints)` that calls `call_api` for each endpoint concurrently, and returns the endpoints whose response ends with `200`, in input order.
3. Write a function that appends to a shared list from five threads, using a lock, and check the final length.

## Answers

### Question 1

An HTTP call spends most of its time waiting, and a waiting thread releases the GIL, so the threads overlap their waits. Parsing numbers is pure Python work, which needs the GIL, so the threads take turns. Use processes for that work.

### Question 2

In [ ]:
async def first_ok(endpoints):
    responses = await asyncio.gather(*(call_api(e) for e in endpoints))
    return [e for e, r in zip(endpoints, responses) if r.endswith("200")]

assert await first_ok(["/a", "/b"]) == ["/a", "/b"]

### Question 3

In [ ]:
shared = []
shared_lock = threading.Lock()

def append_many(n):
    for i in range(n):
        with shared_lock:
            shared.append(i)

workers = [threading.Thread(target=append_many, args=(1_000,)) for _ in range(5)]
for worker in workers:
    worker.start()
for worker in workers:
    worker.join()

assert len(shared) == 5_000

## Interview notes

- **Threads for waiting, processes for computing, async for very many waits.** Pick by what the work spends its time on.
- **The GIL limits Python bytecode, not your data.** NumPy and Polars release it during heavy native computation.
- **`pool.map` keeps input order; `as_completed` gives completion order.** Choose the one your downstream step needs.
- **`counter += 1` is not atomic across threads.** Use a lock, or avoid sharing mutable state.
- **Bound concurrency.** A semaphore or a fixed pool size protects the system you call.
- **An `async` function does nothing until it is awaited.** Forgetting `await` is a common bug, and it produces a warning instead of an error.

## Structured concurrency and deadlines

The Python track targets Python 3.12+. TaskGroup waits for its tasks and cancels siblings when one fails; failures propagate as an ExceptionGroup. asyncio.timeout bounds a block and raises TimeoutError outside it. Free-threaded CPython builds are an optional runtime choice, not a reason to remove locks: shared invariants still require synchronization. Check extension compatibility before choosing such a build.

[Python asyncio tasks](https://docs.python.org/3.12/library/asyncio-task.html)

In [ ]:
import asyncio

async def task_group_demo():
    async def twice(value):
        await asyncio.sleep(0)
        return value*2
    async with asyncio.TaskGroup() as group:
        tasks=[group.create_task(twice(i)) for i in range(3)]
    assert [task.result() for task in tasks]==[0,2,4]
    try:
        async with asyncio.timeout(0.01):
            await asyncio.sleep(10)
    except TimeoutError:
        return 'deadline enforced'
    raise AssertionError('expected timeout')

assert await task_group_demo() == 'deadline enforced'